# 4.2 · 回归诊断 / Regression Diagnostics

> **课程定位 / Where this fits**
> 第 2 课，**Part 4 · 监督学习：回归**。
> Lesson 2, **Part 4 · Supervised Regression**.
>
> 4.1 列了线性回归的五大假设。这一课是回归的"**体检**"：用残差图、QQ 图、VIF、Cook's distance 等工具，逐一检验这些假设到底满不满足，以及不满足时怎么办。**做"解释型"建模(要可信的 p 值/置信区间)时，诊断是必做步骤。**
> 4.1 listed the five assumptions. This lesson is the regression "**health check**": use residual plots, QQ plots, VIF, Cook's distance to verify each assumption and what to do when violated. For **explanatory modeling (trustworthy p-values/CIs), diagnostics are mandatory.**
>
> 💼 **实战/面试视角**："怎么判断线性回归假设满足 / 异方差怎么办 / 多重共线性怎么查" 偏统计/分析岗。
> 💼 **Practical/interview angle:** "how to check assumptions / handle heteroscedasticity / detect multicollinearity" — stats/analytics roles.

> 💡 **面试相关 / Interview-relevant**
> - "残差图怎么看 / 能诊断什么"（出镜率 ★★★★）
> - "什么是异方差 / 怎么检验 / 怎么办"（★★★★★）
> - "多重共线性 / VIF 怎么用"（★★★★★）
> - "R² vs 调整 R²"（★★★★）
> - "高杠杆点 / 影响点 / Cook's distance"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解 **R² vs 调整 R²**，知道选模型看后者。
   Understand R² vs adjusted R²; use the latter for model selection.
2. 用**四张诊断图**检验线性/同方差/正态。
   Use the **four diagnostic plots** to check linearity/homoscedasticity/normality.
3. 用 **Breusch-Pagan 检验**确认异方差，并用**稳健标准误**应对。
   Confirm heteroscedasticity via Breusch-Pagan and handle it with robust SEs.
4. 用 **VIF** 诊断多重共线性。
   Diagnose multicollinearity with VIF.
5. 用**杠杆 / Cook's distance** 找影响点。
   Find influential points with leverage / Cook's distance.

## 目录 / TOC
1. [先建直觉 + R² vs 调整 R² ⭐](#1)
2. [四张诊断图 ⭐](#2)
3. [异方差：检验 + 稳健 SE ⭐](#3)
4. [多重共线性：VIF ⭐](#4)
5. [影响点：杠杆 + Cook's D ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉 + R² vs 调整 R² ⭐ / Intuition & R² vs Adjusted R²

诊断的核心对象是**残差(residual)** = 真实值 − 预测值。如果模型抓住了数据里的所有规律，**残差里应该只剩"纯随机噪声"**——没有任何模式。所以诊断的逻辑就是：**画残差，看它像不像纯噪声**。一旦残差里看出模式（弯曲、喇叭形、偏态），就说明某个假设被违反了。
The central object is the **residual** = actual − predicted. If the model captured all the structure, **the residuals should be pure random noise** — no pattern left. So the logic of diagnostics: **plot residuals and check if they look like noise.** Any pattern (curve, fan shape, skew) means an assumption is violated.

先看一个易混点：**R² 永远随特征增多而升高**——哪怕加一列纯噪声，R² 也只增不减。所以**比较模型时要看调整 R²(adjusted R²)**，它对特征数量加了惩罚，只有"真正有用"的特征才会让它上升。
A quick subtlety: **R² always rises as you add features** — even pure noise never lowers it. So **compare models by adjusted R²**, which penalizes feature count and rises only for genuinely useful features.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import statsmodels.api as sm
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)

from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df = data.frame.sample(3000, random_state=0).reset_index(drop=True)   # 取子样本加速
X = df[["MedInc","HouseAge","AveRooms","AveBedrms","Population","AveOccup"]]
y = df["MedHouseVal"]

X_sm = sm.add_constant(X)            # 加截距列 / add intercept
model = sm.OLS(y, X_sm).fit()
print(f"R² = {model.rsquared:.3f}, 调整 R² adjusted = {model.rsquared_adj:.3f}")
print("💡 R² 永随特征增多而升(即使加噪声列); adjusted R² 惩罚特征数 → 选模型看它")


<a id="2"></a>
## 2. 四张诊断图 ⭐ / The Four Diagnostic Plots

回归诊断有四张"标准体检图"，每张查一个假设：
Four standard diagnostic plots, each checking an assumption:
1. **残差 vs 拟合值**：应是一团**无模式**的云。出现曲线→违反线性；出现**喇叭形**（散布随拟合值变大）→违反同方差。
   **Residuals vs Fitted:** should be a patternless cloud. A curve → nonlinearity; a **fan shape** → heteroscedasticity.
2. **QQ 图**：残差分位 vs 正态分位，应贴对角线（接 2.2）。偏离→残差非正态。
   **QQ plot:** residual quantiles vs normal; should hug the diagonal. Deviation → non-normal residuals.
3. **Scale-Location**：$\sqrt{|标准化残差|}$ vs 拟合，应水平。上升→异方差。
   **Scale-Location:** $\sqrt{|\text{std resid}|}$ vs fitted; should be flat. Rising → heteroscedasticity.
4. **残差直方图**：应近似对称钟形。
   **Residual histogram:** should be roughly symmetric and bell-shaped.


In [ ]:
resid = model.resid                                   # 残差 = 真实 - 预测
fitted = model.fittedvalues                            # 拟合(预测)值
std_resid = model.get_influence().resid_studentized_internal  # 标准化残差

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
# ① 残差 vs 拟合: 检验线性+同方差 / linearity + homoscedasticity
axes[0,0].scatter(fitted, resid, alpha=0.2, s=8); axes[0,0].axhline(0, color="r", ls="--")
axes[0,0].set_xlabel("拟合 fitted"); axes[0,0].set_ylabel("残差 residual")
axes[0,0].set_title("残差 vs 拟合: 应无模式; 这里有喇叭形→异方差!")
# ② QQ 图: 检验残差正态 / normality
st.probplot(std_resid, dist="norm", plot=axes[0,1])
axes[0,1].set_title("QQ 图: 应贴对角线(正态 2.2)")
axes[0,1].get_lines()[0].set(markersize=3, alpha=0.4)
# ③ Scale-Location: 检验同方差(散布是否恒定) / spread vs fitted
axes[1,0].scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.2, s=8)
axes[1,0].set_xlabel("拟合 fitted"); axes[1,0].set_ylabel("√|标准化残差|")
axes[1,0].set_title("Scale-Location: 应水平(上升→异方差)")
# ④ 残差直方图 / residual histogram
axes[1,1].hist(resid, bins=40); axes[1,1].set_xlabel("残差 residual")
axes[1,1].set_title(f"残差分布(偏度 skew={st.skew(resid):.2f})")
plt.tight_layout(); plt.show()
print("结论: 残差 vs 拟合有喇叭形 + 残差右偏 → 违反同方差+正态(房价数据典型问题)")


<a id="3"></a>
## 3. 异方差：检验 + 稳健 SE ⭐ / Heteroscedasticity

**异方差(heteroscedasticity)** = 残差的方差不恒定（如房价越高、预测误差越大）。它**不影响系数的点估计**，但会让普通的标准误、p 值、置信区间**不可信**。
**Heteroscedasticity** = non-constant residual variance (e.g. errors grow with price). It **doesn't bias the coefficient estimates**, but it makes ordinary standard errors, p-values, and CIs **untrustworthy**.

诊断用 **Breusch-Pagan 检验**（p < 0.05 → 确认异方差）。应对：用**稳健标准误(robust/HC3 SE)** 重新算推断——系数不变，但 SE 被修正（这是 2.11 pairs bootstrap 之外的另一条标准路径）。
Test with **Breusch-Pagan** (p < 0.05 → heteroscedastic). Fix: recompute inference with **robust/HC3 standard errors** — coefficients unchanged, SEs corrected (a standard alternative to the pairs bootstrap from 2.11).


In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp = het_breuschpagan(resid, X_sm)    # Breusch-Pagan 异方差检验
print(f"Breusch-Pagan: LM 统计量={bp[0]:.1f}, p={bp[1]:.2e}")
print("→ p < 0.05, 拒绝'同方差'假设, 确认异方差\n")

# 稳健标准误 HC3: 系数不变, 但 SE 在异方差下被修正 / robust SEs
robust = sm.OLS(y, X_sm).fit(cov_type="HC3")
comp = pd.DataFrame({"coef": model.params[:4].round(3),
                     "SE_普通 normal": model.bse[:4].round(4),
                     "SE_稳健 robust": robust.bse[:4].round(4)})
print("普通 SE vs 稳健 SE(HC3) 对比(前4个系数):")
print(comp.to_string())
print("\n系数不变, 但 SE 变了 → 异方差下用稳健 SE 做推断才正确")


<a id="4"></a>
## 4. 多重共线性：VIF ⭐ / Multicollinearity & VIF

**多重共线性** = 特征之间高度相关（如"房间数"和"卧室数"）。它会让系数**极不稳定**（数据稍变，系数大幅甚至变号），p 值也失真——虽然预测可能不受影响，但**系数解读完全不可信**。
**Multicollinearity** = features highly correlated (e.g. "rooms" and "bedrooms"). It makes coefficients **wildly unstable** (small data changes flip their sign) and distorts p-values — predictions may be fine, but **coefficient interpretation becomes meaningless**.

诊断用 **VIF（方差膨胀因子）**：把某特征用其他特征回归，$\text{VIF}=\frac{1}{1-R^2}$。**VIF > 5（或 10）就警惕**。应对：删一个、造比率特征(3.6)、或上 Ridge(4.4，它对共线性稳健)。
Diagnose with **VIF (variance inflation factor)**: regress a feature on the others, $\text{VIF}=\frac{1}{1-R^2}$. **VIF > 5 (or 10) is a red flag.** Fix: drop one, build a ratio feature (3.6), or use Ridge (4.4, robust to collinearity).


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 对每个特征算 VIF: 它能被其他特征"解释"得越多, VIF 越高 / VIF per feature
vif = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
}).sort_values("VIF", ascending=False)
print("各特征 VIF (>5 警惕, >10 严重):")
print(vif.round(2).to_string(index=False))
print("\nAveRooms 和 AveBedrms 共线(房间多卧室也多) → VIF 高")
print("处置: 删一个 / 造比率特征 bedrooms÷rooms(3.6) / 上 Ridge(4.4, 对共线性稳健)")


<a id="5"></a>
## 5. 影响点：杠杆 + Cook's D ⭐ / Influential Points

不是所有异常点都同等危险。**杠杆(leverage)** 衡量一个点的**特征**有多极端；**残差**衡量它被预测得多差。两者都大的点就是**影响点**——它能**单枪匹马地扭曲整条回归线**。**Cook's distance** 把两者合成一个分数，超过 $4/n$ 通常视为可疑。
Not all outliers are equally dangerous. **Leverage** measures how extreme a point's **features** are; the **residual** measures how badly it's predicted. A point high on both is an **influential point** — it can **single-handedly distort the whole regression line**. **Cook's distance** combines both into one score; above $4/n$ is typically flagged.

应对：先查是不是**数据错误**(3.3)；如果是真实点，别删，改用**稳健回归(4.15)**。
Fix: first check if it's a **data error** (3.3); if genuine, don't delete — use **robust regression (4.15)**.


In [ ]:
influence = model.get_influence()
leverage = influence.hat_matrix_diag       # 杠杆: 特征有多极端
cooks = influence.cooks_distance[0]        # Cook's distance: 综合影响力

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# 杠杆 vs 标准化残差: 右上/右下角 = 高杠杆+大残差 = 影响点 / leverage vs residual
axes[0].scatter(leverage, std_resid, alpha=0.3, s=10); axes[0].axhline(0, color="gray", ls="--")
axes[0].set_xlabel("杠杆 leverage"); axes[0].set_ylabel("标准化残差 std residual")
axes[0].set_title("杠杆 vs 残差: 右上/右下角=影响点")
# Cook's distance: 超过 4/n 的点可疑 / stems above 4/n are suspect
axes[1].stem(cooks, markerfmt=",")
axes[1].axhline(4/len(y), color="r", ls="--", label=f"警戒线 4/n={4/len(y):.4f}")
axes[1].set_xlabel("样本索引 sample index"); axes[1].set_ylabel("Cook's D"); axes[1].legend()
axes[1].set_title(f"Cook's distance ({(cooks>4/len(y)).sum()} 个超线)")
plt.tight_layout(); plt.show()
print(f"{(cooks>4/len(y)).sum()} 个高影响点 ({(cooks>4/len(y)).mean():.1%})")
print("处置: 先查是不是数据错误(3.3); 真实点别删, 改用稳健回归(4.15)")


<a id="6"></a>
## 6. 小结 / Summary

```
诊断核心: 残差应像纯噪声, 有模式=假设被违反
R² 永随特征升; 选模型看 adjusted R²(惩罚特征数)
四张图: 残差vs拟合(线性+同方差) / QQ(正态) / Scale-Location(同方差) / 残差直方图
异方差: 残差方差不恒定(喇叭形); Breusch-Pagan 检验; 用稳健SE(HC3)修正推断(系数不变)
多重共线性: 特征高相关→系数不稳; VIF>5/10 警惕; 删/造比率/上 Ridge
影响点: 高杠杆(特征极端)+大残差; Cook's D>4/n; 先查数据错, 真实点用稳健回归(4.15)
```

### 💡 面试速查 / Interview cheat-sheet
1. **残差有模式 = 假设被违反**；残差 vs 拟合图最常用。
   Patterned residuals = violated assumptions; residual-vs-fitted is the workhorse plot.
2. **异方差**(喇叭形): Breusch-Pagan 检验, 用**稳健 SE**修正(系数不变)。
   Heteroscedasticity (fan): Breusch-Pagan test, fix with robust SEs (coefficients unchanged).
3. **VIF > 5/10 = 多重共线性**: 删特征/造比率/Ridge。
   VIF > 5/10 = multicollinearity: drop/ratio/Ridge.
4. **调整 R²** 才能公平比较不同特征数的模型。
   Use adjusted R² to fairly compare models with different feature counts.
5. **Cook's distance** 找影响点; 真实点别删, 用稳健回归。
   Cook's distance finds influential points; keep genuine ones, use robust regression.

### 下一节 / Next
**4.3 多项式回归**——线性模型只能画直线/平面。多项式回归通过加高次项让它拟合曲线，同时引出过拟合与偏差-方差权衡。
**4.3 Polynomial Regression** — linear models only fit lines/planes. Polynomial regression adds higher-order terms to fit curves, introducing overfitting and the bias-variance trade-off.
